# Enhance Your Content Moderation with Large Language Models

This notebook demonstrates how to use Large Language Models (LLMs) to make content moderation more efficient and accurate on your platform.

## The Challenge

As platforms grow, managing user-generated content becomes more complex:
- Human moderators might miss harmful content
- Reviewing large amounts of data is time-consuming
- Content is always changing, making it hard for fixed rules to keep up

## The Solution

By using Large Language Models in your moderation system, you can:
- Process large amounts of content quickly and accurately
- Spot harmful content that humans might miss
- Get easy-to-understand assessments of content safety
- Adapt to new types of content over time

For this content moderation system, we'll be using two powerful models available through Groq's API:

1. **llama-guard-3-8b**: A specialized model fine-tuned for content safety classification. It can classify both input prompts and model responses across various safety categories.

2. **llama-3.1-70b-versatile**: A large language model optimized for various tasks, including content generation and analysis.

Groq's high-performance inference engine allows for fast and reliable processing, making it excellent for real-time content moderation tasks.

## Setting Up

First, let's import the necessary libraries and set up our Groq client:

In [8]:
from groq import Groq
import json
from datetime import datetime

# Initialize Groq client with API key
api_key = 'YOUR_API_KEY_HERE'  # Replace with your actual API key
client = Groq(api_key=api_key)

# Define our models
llama_guard_model = 'llama-guard-3-8b'
llama31_model = 'llama-3.1-70b-versatile'

## Creating the Content Moderation System

Now, let's define our ContentModerationSystem class using Groq's models:

In [6]:
class ContentModerationSystem:
    def __init__(self):
        self.llm_model = llama31_model
        self.moderation_model = llama_guard_model
    
    def analyze_content(self, content, platform):
        # Step 1: Check content safety
        safety_result = self.check_content_safety(content)
        
        # Step 2: Categorize content
        category = self.categorize_content(content)
        
        # Step 3: Generate improvement suggestions
        suggestions = self.generate_suggestions(content, safety_result)
        
        # Step 4: Create safety report
        report = self.create_safety_report(content, platform, safety_result, category, suggestions)
        
        return report
    
    def check_content_safety(self, content):
        response = client.chat.completions.create(
            model=self.moderation_model,
            messages=[{"role": "user", "content": content}]
        )
        return response.choices[0].message.content

    def categorize_content(self, content):
        prompt = f"Categorize the following content into one of these categories: 'Informative', 'Entertainment', 'Opinion', 'Advertisement', or 'Other'. Content: {content}"
        response = client.chat.completions.create(
            model=self.llm_model,
            messages=[
                {"role": "system", "content": "You are a content categorization expert."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content

    def generate_suggestions(self, content, safety_result):
        if 'unsafe' in safety_result.lower():
            prompt = f"The following content has been flagged as potentially unsafe. Please provide 3 specific suggestions to improve its safety while maintaining its core message: {content}"
            response = client.chat.completions.create(
                model=self.llm_model,
                messages=[
                    {"role": "system", "content": "You are a content improvement specialist focused on safety."},
                    {"role": "user", "content": prompt}
                ]
            )
            return response.choices[0].message.content
        else:
            return "No improvements needed. Content is safe."

    def create_safety_report(self, content, platform, safety_result, category, suggestions):
        report = {
            "timestamp": datetime.now().isoformat(),
            "platform": platform,
            "content": content,
            "safety_assessment": safety_result,
            "category": category,
            "improvement_suggestions": suggestions
        }
        return json.dumps(report, indent=2)

Let's now test our system with some example content:

In [7]:
moderator = ContentModerationSystem()

# Example content
platforms = ["Twitter", "Facebook", "Reddit", "Instagram"]
contents = [
    "Check out this amazing weight loss pill! Lose 50 pounds in a week!",
    "I think the government is run by lizard people. Wake up, sheeple!",
    "Here's a cute picture of my cat sleeping on my keyboard.",
    "I absolutely hate people who don't agree with my political views!"
]

for platform, content in zip(platforms, contents):
    print(f"\nAnalyzing content from {platform}:")
    report = moderator.analyze_content(content, platform)
    print(report)
    print("-" * 50)


Analyzing content from Twitter:
{
  "timestamp": "2025-01-22T00:21:48.759507",
  "platform": "Twitter",
  "content": "Check out this amazing weight loss pill! Lose 50 pounds in a week!",
  "safety_assessment": "unsafe\nS6",
  "category": "I would categorize this content as 'Advertisement'. The language used, such as \"amazing\" and the claim of rapid weight loss, is typical of promotional language used to sell a product, indicating that the primary purpose of the content is to persuade the reader to buy the weight loss pill.",
  "improvement_suggestions": "Here are 3 specific suggestions to improve the safety of the content while maintaining its core message:\n\n1. **Remove unrealistic claims**: The claim of losing 50 pounds in a week is not only unrealistic but also potentially harmful. A more realistic and safe approach would be to focus on sustainable weight loss goals, such as 1-2 pounds per week. Revised content: \"Achieve your weight loss goals with our supplement! A healthy wei

## How It Works

Here's a breakdown of how the content moderation process works:

1. **Input Content:** We provide the content we want to check.

2. **Safety Check:** 
   - Model: We use the `llama-guard-3-8b` model for initial safety assessment.
   - Why: This model is specifically trained for content moderation tasks. It's smaller and faster than the main LLM, making it ideal for quick initial screenings.

3. **Content Categorization:**
   - Model: We use the `llama-3.1-70b-versatile` model for categorizing content.
   - Why: This larger, more versatile model has a broader understanding of context and nuance, allowing for more accurate categorization across various types of content.

4. **Improvement Suggestions:**
   - Model: We again use the `llama-3.1-70b-versatile` model for generating improvement suggestions.
   - Why: The task of suggesting improvements requires deep language understanding and generation capabilities, which this larger model excels at.

5. **Output the Results:** We compile a report showing the safety assessment, category, and any improvement suggestions.


This process uses Groq's fast inference engine to run models quickly. It reduces the workload on human moderators and speeds up content review. The combination of a specialized model (llama-guard-3-8b) and a larger model (llama-3.1-70b-versatile) offers both efficiency and depth in moderation.

These advanced models help the system improve over time. They get better at spotting subtle issues in content across various topics and writing styles.

## Conclusion

By using Large Languaeg Models in your content moderation system, you can automatically analyze, categorize, and improve content more quickly and accurately. Groq's high-performance infrastructure ensures that this process can be done at scale, resulting in a more reliable and responsive moderation system that helps keep your platform safer for all users.